In [ ]:
import torchvision.transforms as tf

import src.resnet_subtype.modality_ensemble as modality_ensemble
import src.resnet_subtype.transform as transform

In [ ]:
TRANSFORM_VAL_2D = tf.Compose([
    transform.ResizeWithConstantRatio(target_size=128),
])
TRANSFORM_VAL_3D = TRANSFORM_VAL_2D

In [ ]:
BEST_RESNET_MODELS = {"DCE-MRI":
                          {"channel_couple": [0, 1, 2, 3, 4, 5, 6],
                           "model_id": "2025-09-14_17:11:19"},
                      "T2W":
                          {"channel_couple": [7],
                           "model_id": "2025-09-08_15:09:47"},
                      "ADC":
                          {"channel_couple": [8],
                           "model_id": "2025-09-09_06:29:03"},
                      "PET":
                          {"channel_couple": [9],
                           "model_id": "2025-09-15_18:49:00"}
                      }

In [ ]:
file_task = "../data/006_LumA_LumB_BBox_liminal_tvt_prior.json"
dir_models = "../models"

####

alpha = 0.0
C = 1.0
eps_preds = 0
eps_labels = 1e-2
ensemble_selection = "reg_linear"
load_best_model_info = BEST_RESNET_MODELS

criterion = ["patient", "bce"]
axial_only = True
transform_2D = TRANSFORM_VAL_2D
transform_3D = TRANSFORM_VAL_3D

####

modality_idxs_per_version = {"T1W": [0, 1, 2, 3, 4, 5, 6],
                             "T2W": [7],
                             "ADC": [8],
                             "PET": [9]}

In [ ]:
# Selecting the optimal ensemble weights.
se = modality_ensemble.SelectEnsemble(file_task=file_task,
                                      dir_base=dir_models,
                                      criterion=criterion,
                                      modality_idxs_per_version=modality_idxs_per_version,
                                      method_ensemble_selection=ensemble_selection,
                                      alpha=alpha,
                                      C=C,
                                      eps_preds=eps_preds,
                                      eps_labels=eps_labels,
                                      axial_only=axial_only,
                                      transform_2D=transform_2D,
                                      transform_3D=transform_3D,
                                      load_best_model_info=load_best_model_info)

In [ ]:
# ResNet ensemble prediction and evaluation on test dataset.
se.predict_evaluate(tvt="test",
                    verbose=True)